<small>**Introduction to Statistical Learning with Python** · BSc Data Science, EBS Universität · Fall Term 2026 · Instructor: Constantin Lisson</small>

# Linear Models and Regularization Methods

<a target="_blank" href="https://colab.research.google.com/github/lissonc/islp/blob/2026-fall/Ch06-varselect-lab.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
# --- Google Colab setup: run this cell first, every session ---
# Installs what this lab needs (takes 1-2 minutes). It does nothing
# when the notebook runs outside Colab. See Ch00-course-introduction.
import sys
if "google.colab" in sys.modules:
    %pip install -q ISLP==0.4.1

In this lab we implement many of the techniques discussed in this chapter.
We import some of our libraries at this top
level. 

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
from statsmodels.api import OLS
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from functools import partial

We again collect the new imports needed for this lab.

In [ ]:
import itertools
import math
import time
from sklearn.pipeline import Pipeline
from ISLP.models import \
     (Stepwise,
      sklearn_selected)

## Subset Selection Methods
Here we implement methods that reduce the number of parameters in a
model by restricting the model to a subset of the input variables.
To choose among the candidate models, we use *indirect* estimates of
the test error: $C_p$, AIC, BIC and adjusted $R^2$, which adjust the
training error for the size of the model (Section 6.1.3).

We will apply these methods to the `Hitters`
data.  We wish to predict a baseball player’s `Salary` on the
basis of various statistics associated with performance in the
previous year.

First of all, we note that the `Salary` variable is missing for
some of the players: `np.isnan()` marks the missing entries, and `sum()` counts them.

In [ ]:
Hitters = load_data('Hitters')
np.isnan(Hitters['Salary']).sum()

We see that `Salary` is missing for 59 players. The `dropna()` method removes all rows with missing values.

In [ ]:
Hitters = Hitters.dropna();
Hitters.shape

ℹ️ A note on terminology: *predictors*, *features*, *independent variables*,
*input variables* and *exogenous variables* all mean the same thing. So do
*response*, *dependent variable*, *output variable* and *endogenous variable*.

Both selection methods below work with the model matrix `X`, which
contains an intercept column and the 19 predictors (the categorical
variables `League`, `Division` and `NewLeague` become dummy variables).
We also need an estimate of the residual variance $\sigma^2$ for $C_p$.
We fit the biggest model, using all the variables, and estimate
$\sigma^2$ based on its MSE.

In [ ]:
design = MS(Hitters.columns.drop('Salary')).fit(Hitters)
Y = np.array(Hitters['Salary'])
X = design.transform(Hitters)
sigma2 = OLS(Y,X).fit().scale
X.columns

### Best Subset Selection
Best subset selection (Algorithm 6.1) fits a separate least squares
regression for every possible combination of the $p=19$ predictors.
For each model size $k$, it keeps the best model $\mathcal{M}_k$: the
one with the smallest RSS. We write this ourselves.

The function below fits one model, always including the intercept, and
returns its RSS. It works on `X_np`, a NumPy array version of `X`, and
looks up the column number of each feature in the dictionary `column`.

In [ ]:
X_np = X.to_numpy(dtype=float)
column = {name: j for j, name in enumerate(X.columns)}

def process_feature_set(Y, X_np, feature_set):
    "Fit least squares on the intercept plus `feature_set`; return the RSS"
    Xk = X_np[:, [column['intercept']] + [column[f] for f in feature_set]]
    beta_hat = np.linalg.lstsq(Xk, Y, rcond=None)[0]
    RSS = np.sum((Y - Xk @ beta_hat)**2)
    return {'features': feature_set, 'RSS': RSS}

ℹ️ We compute the least squares fit directly with `np.linalg.lstsq()`
instead of `OLS()`, and select columns from an array rather than a data
frame. This gives the same coefficients, but much faster, which matters
here: we are about to fit more than half a million models, and with
`OLS()` this would take several minutes.

The next function fits all $\binom{p}{k}$ models with exactly $k$
predictors and returns the best one. `itertools.combinations()`
generates the feature sets, and `time.time()` lets us track how long
it takes.

In [ ]:
predictors = X.columns.drop('intercept')

def select_best(Y, X_np, k):
    "Fit all models with k predictors; return the one with the smallest RSS"
    tic = time.time()
    results = [process_feature_set(Y, X_np, feature_set)
               for feature_set in itertools.combinations(predictors, k)]
    models = pd.DataFrame(results)
    best_model = models.loc[models['RSS'].idxmin()]
    toc = time.time()
    print(f"Processed {models.shape[0]} models with {k} predictors in {toc-tic:.2f} seconds.")
    return best_model

#### How many models are there?
Before we run the search, let’s count the models. With just three
predictors, `itertools.combinations()` lists all feature sets of each
size $k$:

In [ ]:
small = ['Hits', 'Walks', 'Years']
for k in range(len(small)+1):
    subsets = list(itertools.combinations(small, k))
    print(f"k={k}: {len(subsets)} models {subsets}")

There are $\binom{p}{k}$ ways to choose $k$ out of $p$ predictors
(`math.comb(p, k)`): here $1+3+3+1 = 8 = 2^3$. In general, each
predictor is either in or out of the model, so there are $2^p$ models in
total, including the null model $\mathcal{M}_0$, which contains only the
intercept. For our $p=19$ predictors:

In [ ]:
n_models = pd.Series([math.comb(len(predictors), k)
                      for k in range(len(predictors)+1)],
                     name='# models')
n_models.index.name = 'k'
n_models

In [ ]:
n_models.sum(), 2**len(predictors)

More than half a million models, with up to 92,378 of them for a single
model size. The number doubles with every additional predictor: with
$p=40$ predictors, there would be about $10^{12}$ models.

We now run `select_best()` for every $k$ from 1 to 19 and collect
$\mathcal{M}_1, \dots, \mathcal{M}_{19}$ in the data frame
`models_best`. This takes a little while (about half a minute).

In [ ]:
best = []
tic = time.time()
for k in range(1, len(predictors)+1):
    best.append(select_best(Y, X_np, k))
toc = time.time()
print(f"Total elapsed time: {toc-tic:.1f} seconds.")

models_best = pd.DataFrame(best, index=range(1, len(predictors)+1))
models_best.index.name = 'k'
models_best

Each row is the best model of its size. For example, the best
two-predictor model contains only `Hits` and `CRBI`. To look at a
model in detail, we refit it with `OLS()`.

In [ ]:
def fit_OLS(features):
    return OLS(Y, X[['intercept'] + list(features)]).fit()

print(fit_OLS(models_best.loc[2, 'features']).summary())

The RSS of $\mathcal{M}_k$ falls with every predictor we add, so it
cannot tell us which of $\mathcal{M}_1, \dots, \mathcal{M}_{19}$ to
pick: it measures training error, not test error. Instead we compute
four indirect estimates of the test error. $C_p$ follows (6.2), with the
$\hat{\sigma}^2$ we estimated above; AIC, BIC and adjusted $R^2$ are
attributes of a fitted `OLS()` model. (`statsmodels` computes AIC and
BIC from the log-likelihood, so their values differ from (6.2) and (6.3)
by constants, but they rank the models in the same way.)

In [ ]:
n = X.shape[0]
k = np.asarray(models_best.index)
fits = [fit_OLS(features) for features in models_best['features']]
criteria = pd.DataFrame({'Cp': (models_best['RSS'] + 2 * k * sigma2) / n,
                         'AIC': [fit.aic for fit in fits],
                         'BIC': [fit.bic for fit in fits],
                         'Adjusted R2': [fit.rsquared_adj for fit in fits]},
                        index=models_best.index)
criteria

We plot the RSS and the four criteria against the number of predictors.
For each criterion we mark the best model: the smallest $C_p$, AIC and
BIC, and the largest adjusted $R^2$. (We mark nothing for the RSS, which
is always smallest for the largest model.)

In [ ]:
plot_data = pd.concat([models_best['RSS'].astype(float), criteria], axis=1)
crit_fig, axes = subplots(2, 3, figsize=(18,10))
for ax, name in zip(axes.flat, plot_data.columns):
    values = plot_data[name]
    ax.plot(values.index, values, marker='o')
    if name != 'RSS':
        best_k = values.idxmax() if name == 'Adjusted R2' else values.idxmin()
        ax.plot(best_k, values[best_k], 'or', markersize=12)
    ax.set_xlabel('# predictors', fontsize=15)
    ax.set_xticks(range(1, 20, 2))
    ax.set_ylabel(name, fontsize=15)
axes.flat[-1].axis('off');

The criteria do not agree completely. $C_p$ and AIC choose the model
with 10 predictors and adjusted $R^2$ the one with 11, while BIC, which
places a heavier penalty on models with many predictors, chooses a much
smaller model with 6 predictors. Note also how flat all four curves are
around their optimum: models with somewhat fewer or more predictors are
almost as good. Here are the predictors in the model chosen by BIC.

In [ ]:
models_best.loc[criteria['BIC'].idxmin(), 'features']

### Forward Stepwise Selection
Best subset selection had to fit $2^{19}-1 = 524{,}287$ models. *Forward
stepwise selection* (Algorithm 6.2) is a greedy alternative: it starts
from the null model and, at each step, adds the one predictor that
improves the fit the most. We write it ourselves first, reusing
`process_feature_set()`. The function below takes the predictors already
in the model, tries adding each of the remaining ones, and returns the
best of these models.

In [ ]:
def forward(Y, X_np, selected):
    "Add each remaining predictor to `selected` in turn; return the best model"
    tic = time.time()
    remaining = [p for p in predictors if p not in selected]
    results = [process_feature_set(Y, X_np, selected + (p,))
               for p in remaining]
    models = pd.DataFrame(results)
    best_model = models.loc[models['RSS'].idxmin()]
    toc = time.time()
    print(f"Processed {models.shape[0]} models with {len(selected)+1} predictors in {toc-tic:.4f} seconds.")
    return best_model

Starting from the null model (no predictors), we call `forward()` 19
times, each time keeping the predictors it has selected so far.

In [ ]:
fwd = []
selected = ()
tic = time.time()
for k in range(1, len(predictors)+1):
    best_model = forward(Y, X_np, selected)
    selected = best_model['features']
    fwd.append(best_model)
toc = time.time()
print(f"Total elapsed time: {toc-tic:.2f} seconds.")

models_fwd = pd.DataFrame(fwd, index=range(1, len(predictors)+1))
models_fwd.index.name = 'k'
models_fwd

Forward stepwise selection took a fraction of a second, compared with
about half a minute for best subset selection. It fit
$19 + 18 + \dots + 1 = 190$ models, plus the null model, instead of
$2^{19}$:

In [ ]:
n_models_fwd = 1 + sum(len(predictors) - j for j in range(len(predictors)))
n_models_fwd

The price is that the search is *greedy*: once a predictor is in the
model, it stays there, so forward stepwise selection is not guaranteed
to find the best model of each size. Let’s compare the RSS of the
models found by the two methods.

In [ ]:
pd.DataFrame({'best subset': models_best['RSS'].astype(float),
              'forward stepwise': models_fwd['RSS'].astype(float)})

For $k=1$ to $6$ the two methods find the same models. For $k=7$ and $8$,
however, forward stepwise selection ends up with a larger RSS: it misses
the best model of these sizes. From $k=9$ on, the two agree again.

As for best subset selection, we choose among the models on the forward
path with $C_p$:

In [ ]:
k_fwd = np.asarray(models_fwd.index)
Cp_fwd = (models_fwd['RSS'] + 2 * k_fwd * sigma2) / n
Cp_fwd.idxmin(), sorted(models_fwd.loc[Cp_fwd.idxmin(), 'features'])

#### Forward Selection with `ISLP`
The `ISLP` package provides tools that carry out such searches for us.
We again choose the best model along the forward path based on $C_p$ (6.2).
This score is not built in as a metric to `sklearn`. We therefore
define a function to compute it ourselves, and use it as a scorer. By
default, `sklearn` tries to maximize a score, hence our scoring
function computes the negative $C_p$ statistic.

In [ ]:
def nCp(sigma2, estimator, X, Y):
    "Negative Cp statistic"
    n, p = X.shape
    Yhat = estimator.predict(X)
    RSS = np.sum((Y - Yhat)**2)
    return -(RSS + 2 * p * sigma2) / n 

The first argument of `nCp()` is the estimate of $\sigma^2$ from above.
The function `sklearn_selected()` expects a scorer with just three
arguments --- the last three in the definition of `nCp()`. We use the
function `partial()` (see the Ch02 lab) to freeze the first argument
with our estimate of $\sigma^2$.

In [ ]:
neg_Cp = partial(nCp, sigma2)

We can now use `neg_Cp()` as a scorer for model selection.

Along with a score we need to specify the search strategy. This is done through the object
`Stepwise()`  in the `ISLP.models` package. The method `Stepwise.first_peak()`
runs forward stepwise until any further additions to the model do not result
in an improvement in the evaluation score. Similarly, the method `Stepwise.fixed_steps()`
runs a fixed number of steps of stepwise search.

In [ ]:
strategy = Stepwise.first_peak(design,
                               direction='forward',
                               max_terms=len(design.terms))

We now fit a linear regression model with `Salary` as outcome using forward
selection. To do so, we use the function `sklearn_selected()`  from the `ISLP.models` package. This takes
a model from `statsmodels` along with a search strategy and selects a model with its
`fit` method. Without specifying a `scoring` argument, the score defaults to MSE, and so all 19 variables will be
selected.

In [ ]:
hitters_MSE = sklearn_selected(OLS,
                               strategy)
hitters_MSE.fit(Hitters, Y)
hitters_MSE.selected_state_

Using `neg_Cp` results in a smaller model, as expected, with just 10
variables selected. Compare them with the 10 predictors of the best
subset model chosen by $C_p$ above: in this case, forward stepwise
selection finds the same model (the dummy variable `Division[W]`
appears simply as `Division` here), and so did our own forward search.

In [ ]:
hitters_Cp = sklearn_selected(OLS,
                               strategy,
                               scoring=neg_Cp)
hitters_Cp.fit(Hitters, Y)
hitters_Cp.selected_state_

In [ ]:
sorted(models_best.loc[criteria['Cp'].idxmin(), 'features'])

## Ridge Regression and the Lasso
We will use the `sklearn.linear_model` package (for which
we use `skl` as shorthand below) to fit ridge and lasso regularized
linear models on the `Hitters` data, with the classes `skl.Ridge()` and
`skl.Lasso()` (and their cross-validating versions `skl.RidgeCV()` and
`skl.LassoCV()`). `sklearn` fits the intercept separately and does not
penalize it, so we start from a model matrix without the intercept column.

In [ ]:
D = design.fit_transform(Hitters)
D = D.drop('intercept', axis=1)
X = np.asarray(D)

Note that this replaces the data frame `X` from the subset selection
part with a plain array of the 19 predictors.

### Ridge Regression
Ridge regression minimizes $\text{RSS} + \lambda \sum_{j=1}^p \beta_j^2$,
see (6.5). It is good practice to standardize the columns of `X`
in this application, since the variables are measured in different
units and the penalty treats all coefficients alike. `skl.Ridge()` does
no standardization, so we have to take care of that ourselves. Since we
standardize first, in order to find coefficient estimates on the
original scale, we must *unstandardize* the coefficient estimates.

ℹ️ The tuning parameter $\lambda$ in (6.5) is called `alpha` in
`sklearn`; in `skl.Ridge()` it is exactly the same quantity. In order
to be consistent with the rest of this chapter, we use `lambdas`
rather than `alphas` in what follows. (The book’s own code uses
`skl.ElasticNet()` instead, a more general method that mixes the ridge
and lasso penalties.)

In [ ]:
Xs = X - X.mean(0)[None,:]
X_scale = X.std(0)
Xs = Xs / X_scale[None,:]
lambdas = 10**np.linspace(8, -2, 100)
coefs = []
for lam in lambdas:
    ridge = skl.Ridge(alpha=lam)
    ridge.fit(Xs, Y)
    coefs.append(ridge.coef_)
soln_array = np.array(coefs).T
soln_array.shape

We fit ridge regression once for each value in a grid of 100 values of
$\lambda$, ranging from $\lambda=10^{8}$ to $\lambda=10^{-2}$. This
essentially covers the full range of scenarios from the null model
containing only the intercept, to the least squares fit.

Associated with each value of $\lambda$ is a vector of ridge
regression coefficients, which we collect as a column of
`soln_array`. In this case, `soln_array` is a $19 \times 100$ matrix,
with 19 rows (one for each predictor) and 100 columns (one for each
value of $\lambda$).

We transpose this matrix and turn it into a data frame to facilitate viewing and plotting.

In [ ]:
soln_path = pd.DataFrame(soln_array.T,
                         columns=D.columns,
                         index=-np.log(lambdas))
soln_path.index.name = 'negative log(lambda)'
soln_path

We plot the paths to get a sense of how the coefficients vary with $\lambda$.
To control the location of the legend we first set `legend` to `False` in the
plot method, adding it afterward with the `legend()` method of `ax`.

In [ ]:
path_fig, ax = subplots(figsize=(8,8))
soln_path.plot(ax=ax, legend=False)
ax.set_xlabel(r'$-\log(\lambda)$', fontsize=20)
ax.set_ylabel('Standardized coefficients', fontsize=20)
ax.legend(loc='upper left');

We expect the coefficient estimates to be much smaller, in terms of
$\ell_2$ norm, when a large value of $\lambda$ is used, as compared to
when a small value of $\lambda$ is used. (Recall that the  $\ell_2$ norm is the square root of the sum of squared coefficient values.) We display  the coefficients at the $40$th step,
where $\lambda$ is 1.15e+04.

In [ ]:
beta_hat = soln_path.loc[soln_path.index[39]]
lambdas[39], beta_hat

Let’s compute the $\ell_2$ norm of the standardized coefficients.

In [ ]:
np.linalg.norm(beta_hat)

In contrast, here is the $\ell_2$ norm when $\lambda$ is 109.7.
Note the much larger $\ell_2$ norm of the
coefficients associated with this smaller value of $\lambda$.

In [ ]:
beta_hat = soln_path.loc[soln_path.index[59]]
lambdas[59], np.linalg.norm(beta_hat)

Above we normalized `X` upfront, and fit the ridge model using `Xs`.
The `Pipeline()`  object
in `sklearn` provides a clear way to separate feature
normalization from the fitting of the ridge model itself.

In [ ]:
ridge = skl.Ridge(alpha=lambdas[59])
scaler = StandardScaler(with_mean=True,  with_std=True)
pipe = Pipeline(steps=[('scaler', scaler), ('ridge', ridge)])
pipe.fit(X, Y)

We show that it gives the same $\ell_2$ norm as in our previous fit on the standardized data.

In [ ]:
np.linalg.norm(ridge.coef_)

 Notice that the operation `pipe.fit(X, Y)` above has changed the `ridge` object, and in particular has added attributes such as `coef_` that were not there before.
### Estimating Test Error of Ridge Regression
Choosing an *a priori* value of $\lambda$ for ridge regression is
difficult if not impossible. We will want to use the validation method
or cross-validation to select the tuning parameter. The reader may not
be surprised that the  `Pipeline()` approach can be used in
`skm.cross_validate()` with either a validation method
(i.e. `validation`) or $k$-fold cross-validation. Using the pipeline
ensures that the standardization is learned from the training
observations only.

We fix the random state of the splitters
so that the results obtained will be reproducible. We set up both
a validation split (half of the data for training, half for testing)
and 5-fold cross-validation, and start with $\lambda=4$.

In [ ]:
K = 5
kfold = skm.KFold(K,
                  random_state=0,
                  shuffle=True)
validation = skm.ShuffleSplit(n_splits=1,
                              test_size=0.5,
                              random_state=0)
ridge.alpha = 4
results = skm.cross_validate(pipe,
                             X,
                             Y,
                             scoring='neg_mean_squared_error',
                             cv=validation)
-results['test_score']

The test MSE is 1.274e+05.  Note
that if we had instead simply fit a model with just an intercept, we
would have predicted each test observation using the mean of the
training observations. We can get the same result by fitting a ridge regression model
with a *very* large value of $\lambda$. Note that `1e10`
means $10^{10}$.

In [ ]:
ridge.alpha = 1e10
results = skm.cross_validate(pipe,
                             X,
                             Y,
                             scoring='neg_mean_squared_error',
                             cv=validation)
-results['test_score']

With such a large $\lambda$, the coefficient estimates hardly respond to
the data, and the test MSE is much larger. Obviously choosing $\lambda=4$
is arbitrary, so we will use cross-validation or the validation-set
approach to choose the tuning parameter $\lambda$.
The object `GridSearchCV()`  allows exhaustive
grid search to choose such a parameter.

We first use the validation set method
to choose $\lambda$.

In [ ]:
param_grid = {'ridge__alpha': lambdas}
grid = skm.GridSearchCV(pipe,
                        param_grid,
                        cv=validation,
                        scoring='neg_mean_squared_error')
grid.fit(X, Y)
grid.best_params_['ridge__alpha']
grid.best_estimator_

Alternatively, we can use 5-fold cross-validation.

In [ ]:
grid = skm.GridSearchCV(pipe, 
                        param_grid,
                        cv=kfold,
                        scoring='neg_mean_squared_error')
grid.fit(X, Y)
grid.best_params_['ridge__alpha']
grid.best_estimator_

We now plot the cross-validated MSE as a function of $-\log(\lambda)$, which has shrinkage decreasing from left
to right.

In [ ]:
ridge_fig, ax = subplots(figsize=(8,8))
ax.errorbar(-np.log(lambdas),
            -grid.cv_results_['mean_test_score'],
            yerr=grid.cv_results_['std_test_score'] / np.sqrt(K))
ax.set_ylim([50000,250000])
ax.set_xlabel(r'$-\log(\lambda)$', fontsize=20)
ax.set_ylabel('Cross-validated MSE', fontsize=20);

One can cross-validate different metrics to choose a parameter. The default
metric for `skl.Ridge()` is test $R^2$.
Let’s compare $R^2$ to MSE for cross-validation here.

In [ ]:
grid_r2 = skm.GridSearchCV(pipe, 
                           param_grid,
                           cv=kfold)
grid_r2.fit(X, Y)

Finally, let’s plot the results for cross-validated $R^2$.

In [ ]:
r2_fig, ax = subplots(figsize=(8,8))
ax.errorbar(-np.log(lambdas),
            grid_r2.cv_results_['mean_test_score'],
            yerr=grid_r2.cv_results_['std_test_score'] / np.sqrt(K))
ax.set_xlabel(r'$-\log(\lambda)$', fontsize=20)
ax.set_ylabel('Cross-validated $R^2$', fontsize=20);

### Cross-Validation with `RidgeCV()`
`sklearn` also provides `skl.RidgeCV()`, which performs the search over
$\lambda$ internally: we give it the grid of values and the
cross-validation splits, and it chooses $\lambda$ and refits the model on
the full data. Even with identical splits the results will not agree
*exactly* with our `grid` above, because the standardization of each
feature in `grid` is carried out on each fold, while in `pipeCV` below
it is carried out only once. Nevertheless, the results are similar as
the normalization is relatively stable across folds.

In [ ]:
ridgeCV = skl.RidgeCV(alphas=lambdas,
                      cv=kfold,
                      scoring='neg_mean_squared_error')
pipeCV = Pipeline(steps=[('scaler', scaler),
                         ('ridge', ridgeCV)])
pipeCV.fit(X, Y)
tuned_ridge = pipeCV.named_steps['ridge']
tuned_ridge.alpha_

We see that the value of $\lambda$ that results in the
smallest cross-validation error is 2.66, available as the value `tuned_ridge.alpha_` (the same value that `grid` chose). We mark it in our
earlier plot of the cross-validated MSE. What is the cross-validated
MSE associated with this value of $\lambda$?

In [ ]:
ax = ridge_fig.axes[0]
ax.axvline(-np.log(tuned_ridge.alpha_), c='k', ls='--')
ridge_fig

In [ ]:
-tuned_ridge.best_score_

This represents a further improvement over the test MSE that we got
using $\lambda=4$.  Finally, `tuned_ridge.coef_`
has the coefficients fit on the entire data set
at this value of  $\lambda$.

In [ ]:
tuned_ridge.coef_

As expected, none of the coefficients are zero—ridge regression does
not perform variable selection!

### Evaluating Test Error of Cross-Validated Ridge
Choosing $\lambda$ using cross-validation provides a single regression
estimator, similar to fitting a linear regression model as we saw in
Chapter 3. It is therefore reasonable to estimate what its test error
is. We run into a problem here in that cross-validation will have
*touched* all of its data in choosing $\lambda$, hence we have no
further data to estimate test error. A compromise is to do an initial
split of the data into two disjoint sets: a training set and a test set.
We then fit a cross-validation
tuned ridge regression on the training set, and evaluate its performance on the test set.
We might call this cross-validation nested
within the validation set approach. A priori there is no reason to use
half of the data for each of the two sets in validation. Below, we use
75% for training and 25% for test, with the estimator being ridge
regression tuned using 5-fold cross-validation.  This can be achieved
in code as follows:

In [ ]:
outer_valid = skm.ShuffleSplit(n_splits=1,
                               test_size=0.25,
                               random_state=1)
inner_cv = skm.KFold(n_splits=5,
                     shuffle=True,
                     random_state=2)
ridgeCV = skl.RidgeCV(alphas=lambdas,
                      cv=inner_cv,
                      scoring='neg_mean_squared_error')
pipeCV = Pipeline(steps=[('scaler', scaler),
                         ('ridge', ridgeCV)]);

In [ ]:
results = skm.cross_validate(pipeCV, 
                             X,
                             Y,
                             cv=outer_valid,
                             scoring='neg_mean_squared_error')
-results['test_score']

### The Lasso
We saw that ridge regression with a wise choice of $\lambda$ can
outperform least squares as well as the null model on the
 `Hitters`  data set. We now ask whether the lasso can yield
either a more accurate or a more interpretable model than ridge
regression. The lasso minimizes $\text{RSS} + \lambda \sum_{j=1}^p |\beta_j|$,
see (6.7). We use `skl.LassoCV()`, which, unlike `skl.RidgeCV()`, chooses
its own grid of 100 values of $\lambda$, starting from the smallest value
for which all coefficients are zero. Other than that, we proceed just as
we did in fitting a ridge model.

ℹ️ Careful with the scale of $\lambda$: `skl.Lasso()` and `skl.LassoCV()`
divide the RSS by $2n$, i.e. they minimize
$\frac{1}{2n}\text{RSS} + \alpha \sum_{j=1}^p |\beta_j|$. Their `alpha`
therefore corresponds to $\lambda/(2n)$ in (6.7). This changes the
numbers on the $\lambda$ axis, but not the fitted models.

In [ ]:
lassoCV = skl.LassoCV(cv=kfold)
pipeCV = Pipeline(steps=[('scaler', scaler),
                         ('lasso', lassoCV)])
pipeCV.fit(X, Y)
tuned_lasso = pipeCV.named_steps['lasso']
tuned_lasso.alpha_

To plot the coefficient path of the lasso, we use `skl.Lasso.path()`,
which fits the lasso along a whole grid of $\lambda$ values at once.

In [ ]:
lambdas, soln_array = skl.Lasso.path(Xs, Y)[:2]
soln_path = pd.DataFrame(soln_array.T,
                         columns=D.columns,
                         index=-np.log(lambdas))

We can see from the coefficient plot of the standardized coefficients that depending on the choice of
tuning parameter, some of the coefficients will be exactly equal to
zero.

In [ ]:
path_fig, ax = subplots(figsize=(8,8))
soln_path.plot(ax=ax, legend=False)
ax.legend(loc='upper left')
ax.set_xlabel(r'$-\log(\lambda)$', fontsize=20)
ax.set_ylabel('Standardized coefficients', fontsize=20);

The smallest cross-validated error is lower than the test set MSE of the null model
and of least squares, and very similar to the cross-validated MSE of 115,527 of ridge
regression with $\lambda$ chosen by cross-validation.

In [ ]:
np.min(tuned_lasso.mse_path_.mean(1))

Let’s again produce a plot of the cross-validation error.

In [ ]:
lassoCV_fig, ax = subplots(figsize=(8,8))
ax.errorbar(-np.log(tuned_lasso.alphas_),
            tuned_lasso.mse_path_.mean(1),
            yerr=tuned_lasso.mse_path_.std(1) / np.sqrt(K))
ax.axvline(-np.log(tuned_lasso.alpha_), c='k', ls='--')
ax.set_ylim([50000,250000])
ax.set_xlabel(r'$-\log(\lambda)$', fontsize=20)
ax.set_ylabel('Cross-validated MSE', fontsize=20);

However, the lasso has a substantial advantage over ridge regression
in that the resulting coefficient estimates are sparse. Here we see
that 6 of the 19 coefficient estimates are exactly zero. So the lasso
model with $\lambda$ chosen by cross-validation contains only 13
variables.

In [ ]:
tuned_lasso.coef_

As in ridge regression, we could evaluate the test error
of cross-validated lasso by first splitting into
test and training sets and internally running
cross-validation on the training set. We leave
this as an exercise.